Sekans verilerini işlemek için özel olarak tasarlanmış sinir ağlarıdır. Her zaman adımında, önceki zaman adımındaki bilgiyi saklayarak ve soraki adımlarla bu bilgiyi güncelleyerek çalışırlar.



*   Ağ yapısında gilzi katman vardır burda geçmiş veriler tutulur
*   Zaman boyutunda tekrar yapısı



metin ön işleme(tokenization, padding, etiket kodlama-label encoding)


embedding:word2vec ile sayısal vekötrler oluşturma


RNN modeli oluşturma (embedding-> simpleRNN ->Dense layer)


modelin derlenmesi ve eğitimi


test seti üzerinde modelin


user test (yeni cümlelerin sınıflandırılması için fonksiyon tanımlama)


In [2]:
!pip install gensim tensorflow keras keras_preprocessing "numpy<2.0.0"

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf #derin öğrenme
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer

from keras_preprocessing.sequence import pad_sequences

from gensim.models import Word2Vec #embedding

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [4]:
reviews_data = {
    "text": [
        "Yemekler harikaydı, her şey taze ve lezzetliydi.",
        "Garson çok ilgisizdi, siparişimi unuttular.",
        "Ortam çok şıktı, kesinlikle tekrar geleceğim.",
        "Çorba buz gibi geldi, hiç memnun kalmadım.",
        "Fiyat performans olarak harika bir mekan.",
        "Porsiyonlar çok küçüktü ve doyurucu değildi.",
        "Tatlılara bayıldım, özellikle künefe efsaneydi.",
        "Masalar kirliydi, temizlenmesi için çok bekledik.",
        "Çalışanlar çok güler yüzlü ve misafirperverdi.",
        "Menüdeki yemeklerin çoğu kalmamıştı, hayal kırıklığı.",
        "Deniz manzarası eşliğinde yediğimiz balık şahaneydi.",
        "Hesap beklediğimizden çok daha yüksek geldi, değmez.",
        "Ailecek gittik ve hepimiz çok memnun ayrıldık.",
        "Müzik sesi çok yüksekti, yanımızdakini duyamadık.",
        "Hızlı servis ve lezzetli yemekler, tebrikler.",
        "Etler çok sertti ve çiğnenmiyordu.",
        "İkram ettikleri çay ve meyve tabağı çok ince bir düşünceydi.",
        "Vale hizmeti tam bir fiyaskoydu, arabamı geç getirdiler.",
        "Vegan seçenekleri çok bol ve inanılmaz lezzetliydi.",
        "Tuvaletler hijyenik değildi, bir daha gitmem.",
        "Kahvaltı tabağı çok çeşitli ve doyurucuydu.",
        "Mezeler bayattı, midemizi bozdu.",
        "Şefin spesiyali tek kelimeyle muazzamdı.",
        "Salatanın içinden taş çıktı, inanamıyorum.",
        "Dekorasyon ve ambiyans çok dinlendirici.",
        "Klima çalışmıyordu, içerisi hamam gibiydi.",
        "Ev yapımı limonataları çok ferahlatıcıydı.",
        "Rezervasyonumuz olmasına rağmen yarım saat kapıda bekletildik.",
        "Hamburgerin köftesi sulu sulu ve tam kıvamında pişmişti.",
        "Patates kızartmaları yanmış ve yağ çekmişti.",
        "Doğum günü kutlamamız için bize çok yardımcı oldular, harika bir ekip.",
        "Garsonlar kendi aralarında yüksek sesle tartışıyordu, rahatsız ediciydi.",
        "Tiramisu şimdiye kadar yediğim en iyi tiramisuydu.",
        "Tavuklar tam pişmemişti, içi hala pembeydi.",
        "Hem göze hem damağa hitap eden harika sunumlar vardı.",
        "Sandalyeler çok rahatsız, uzun süre oturmak imkansız.",
        "Sokak lezzetlerini çok şık ve lezzetli bir şekilde sunmuşlar.",
        "Şarap menüsü çok yetersiz ve seçenekler kısıtlı.",
        "Samimi ve sıcak bir mahalle restoranı, favorim oldu.",
        "Çalan müzikler mekanın tarzıyla hiç uyuşmuyordu.",
        "Suşiler inanılmaz taze ve lezzetliydi, şefin ellerine sağlık.",
        "Pizza hamuru kayış gibiydi, malzemeler eksikti.",
        "Uygun fiyata bu kadar kaliteli yemek yemek şaşırttı.",
        "İçeceklere bu kadar fahiş fiyatlar yazılması çok saçma.",
        "Çocuklu aileler için oyun alanı olması büyük avantaj, rahat ettik.",
        "Otopark sorunu var, arabayı bırakacak yer bulana kadar canımız çıktı.",
        "Mantı tıpkı anneannemin yaptığı gibi, bayıldım.",
        "Porsiyonları o kadar küçültmüşler ki, masadan aç kalktık.",
        "Kahveleri çok kaliteli, yemek üstüne harika gitti.",
        "Mekan çok havasız, havalandırma kesinlikle yetersiz."
    ],
    "label": [
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative"
    ]
}

#data frame'e çevir
df= pd.DataFrame(reviews_data)
print(df.head())

                                               text     label
0  Yemekler harikaydı, her şey taze ve lezzetliydi.  positive
1       Garson çok ilgisizdi, siparişimi unuttular.  negative
2     Ortam çok şıktı, kesinlikle tekrar geleceğim.  positive
3        Çorba buz gibi geldi, hiç memnun kalmadım.  negative
4         Fiyat performans olarak harika bir mekan.  positive


In [5]:
#metin ön işleme ve tokenization
tokenizer = Tokenizer() #kelimeleri sayısal indexlere çeviren yapı
tokenizer.fit_on_texts(df["text"]) #kelime sözlüğünü oluştur
text_sequences=tokenizer.texts_to_sequences(df["text"]) #yorumları sayı dizisine çevir
word_index=tokenizer.word_index #sözlük: kelime -index
print(word_index)

{'çok': 1, 've': 2, 'bir': 3, 'kadar': 4, 'harika': 5, 'lezzetliydi': 6, 'için': 7, 'tam': 8, 'yemek': 9, 'yemekler': 10, 'taze': 11, 'kesinlikle': 12, 'gibi': 13, 'geldi': 14, 'hiç': 15, 'memnun': 16, 'mekan': 17, 'değildi': 18, 'bayıldım': 19, 'daha': 20, 'yüksek': 21, 'lezzetli': 22, 'tabağı': 23, 'inanılmaz': 24, 'şefin': 25, 'çıktı': 26, 'gibiydi': 27, 'sulu': 28, 'rahatsız': 29, 'hem': 30, 'yetersiz': 31, 'bu': 32, 'kaliteli': 33, 'harikaydı': 34, 'her': 35, 'şey': 36, 'garson': 37, 'ilgisizdi': 38, 'siparişimi': 39, 'unuttular': 40, 'ortam': 41, 'şıktı': 42, 'tekrar': 43, 'geleceğim': 44, 'çorba': 45, 'buz': 46, 'kalmadım': 47, 'fiyat': 48, 'performans': 49, 'olarak': 50, 'porsiyonlar': 51, 'küçüktü': 52, 'doyurucu': 53, 'tatlılara': 54, 'özellikle': 55, 'künefe': 56, 'efsaneydi': 57, 'masalar': 58, 'kirliydi': 59, 'temizlenmesi': 60, 'bekledik': 61, 'çalışanlar': 62, 'güler': 63, 'yüzlü': 64, 'misafirperverdi': 65, 'menüdeki': 66, 'yemeklerin': 67, 'çoğu': 68, 'kalmamıştı': 69,

In [7]:
#padding
max_sequence_length=max(len(seq) for seq in text_sequences)
print(f"max_sequence_length: {max_sequence_length}")
X=pad_sequences(text_sequences, maxlen=max_sequence_length) #tüm cümelleri aynı uzunluğa getir, eksik kısımları 0 ile doldur
print(f"Giriş verisinin boyutu: {X.shape}") #(50, 11)
print(X)

max_sequence_length: 11
Giriş verisinin boyutu: (50, 11)
[[  0   0   0   0  10  34  35  36  11   2   6]
 [  0   0   0   0   0   0  37   1  38  39  40]
 [  0   0   0   0   0  41   1  42  12  43  44]
 [  0   0   0   0  45  46  13  14  15  16  47]
 [  0   0   0   0   0  48  49  50   5   3  17]
 [  0   0   0   0   0  51   1  52   2  53  18]
 [  0   0   0   0   0   0  54  19  55  56  57]
 [  0   0   0   0   0  58  59  60   7   1  61]
 [  0   0   0   0   0  62   1  63  64   2  65]
 [  0   0   0   0   0  66  67  68  69  70  71]
 [  0   0   0   0   0  72  73  74  75  76  77]
 [  0   0   0   0  78  79   1  20  21  14  80]
 [  0   0   0   0  81  82   2  83   1  16  84]
 [  0   0   0   0   0  85  86   1  87  88  89]
 [  0   0   0   0   0  90  91   2  22  10  92]
 [  0   0   0   0   0   0  93   1  94   2  95]
 [  0  96  97  98   2  99  23   1 100   3 101]
 [  0   0   0 102 103   8   3 104 105 106 107]
 [  0   0   0   0 108 109   1 110   2  24   6]
 [  0   0   0   0   0 111 112  18   3  20 113]
 [ 

In [8]:
#label encoding
label_encoder=LabelEncoder()
y=label_encoder.fit_transform(df["label"]) #etiketleri positive=1 negatif=0 sayısallaştırma

#train test split
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2, random_state=42)

In [9]:
#embedding
sentences=[text.split() for text in df['text']]

In [15]:
#word2vec modelini eğit
word2vec_model = Word2Vec(sentences, vector_size=50, window=5, min_count=1)

embedding_dim=50 #her kelime 60 boyutlu vektör ile temsil edilir

#embedding matrisi
embedding_matrix= np.zeros((len(word_index) +1, embedding_dim))
for word, idx in word_index.items():
  if word in word2vec_model.wv:
    embedding_matrix[idx]=word2vec_model.wv[word]
print(f"embedding_matrix: \ {embedding_matrix}")

embedding_matrix: \ [[ 0.          0.          0.         ...  0.          0.
   0.        ]
 [-0.00099949  0.0004614   0.01034942 ...  0.01910859  0.01000662
   0.01839821]
 [-0.01628956  0.00900224 -0.0082408  ... -0.01417844  0.00181546
   0.01277798]
 ...
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.01956275  0.0163228   0.00265455 ...  0.00013704 -0.00474374
   0.01721763]]


<>:11: SyntaxWarning: invalid escape sequence '\ '
<>:11: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_27737/2380928877.py:11: SyntaxWarning: invalid escape sequence '\ '
  print(f"embedding_matrix: \ {embedding_matrix}")


In [20]:
#model oluşturma
model=Sequential()

#embedding katmanı
model.add(Embedding(input_dim=len(word_index) +1, #kelime sayısı +1
                    output_dim=embedding_dim, #embedding boyutu
                    weights=[embedding_matrix], #önceden eğitilmiş word to vec embedding matrisi
                    input_length= max_sequence_length, #cümlelerin uzunluğu
                    trainable=False #embedding ağırlıkları sabit
                    ))
#rnn katmanı: units-> gizli katman sayısı, return sequneces= sadece son çıktıyı return eder
model.add(SimpleRNN(units=50, return_sequences=False))

#output katmanı
model.add(Dense(1, activation="sigmoid"))


In [23]:
#compile
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

#traning
model.fit(
    X_train, y_train,
    epochs=10, #eğitim tekrar sayısı
    batch_size=2, #mini batch boyutu
    validation_data=(X_test, y_test)
)

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.5000 - loss: 0.7088 - val_accuracy: 0.4000 - val_loss: 0.7090
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5750 - loss: 0.6859 - val_accuracy: 0.4000 - val_loss: 0.6964
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7500 - loss: 0.6734 - val_accuracy: 0.4000 - val_loss: 0.6995
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7500 - loss: 0.6571 - val_accuracy: 0.4000 - val_loss: 0.7079
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7250 - loss: 0.6355 - val_accuracy: 0.3000 - val_loss: 0.7181
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7750 - loss: 0.5893 - val_accuracy: 0.3000 - val_loss: 0.7847
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8250 - loss: 0.5053 - val_accuracy: 0.5000 - val_loss: 0.7748
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7500 - loss: 0.4636 - val_accuracy: 0.3000 - val_los

In [24]:
#değerlendirme
test_loss, test_accuracy=model.evaluate(X_test, y_test)
print(f"Test loss: {test_loss}")
print(f"Test accuracy: {test_accuracy}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.7000 - loss: 0.7934
Test loss: 0.7934088706970215
Test accuracy: 0.699999988079071


In [26]:
#yeni cümlelerin sınıflandırılması için fonk
def classify_sentence(sentence):
  """
    yeni cümleyi alır, işleme sokar ve model ile sınıflandırmaya çalışır
  """
  seq=tokenizer.texts_to_sequences([sentence]) #cümleyi sayısal dizilere çevirir
  padded_seq= pad_sequences(seq, maxlen=max_sequence_length) #uzunluğu normalize eder

  prediction=model.predict(padded_seq) #modelden olasılık al
  prediced_class=(prediction >0.5).astype(int) #0.5 üstü positive etiket alır

  label="positive" if prediced_class[0][0] == 1 else "negative"
  return label_encoder

new_sentence="Restoran çok temizdi ve yemekler çok güzeldi"
result=classify_sentence(new_sentence)
print(result)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step
LabelEncoder()
